# 🥉 Notebook 01 — Bronze Layer: Ingest 6 Raw Source Tables

**Goal:** Load all 6 CSV source files into Bronze Delta Tables with full audit metadata. Zero transformation.

> **Run time:** ~3 min

```
Source CSVs
  customers.csv    ──► bronze_customers     (500 rows)
  accounts.csv     ──► bronze_accounts      (500 rows)
  branches.csv     ──► bronze_branches      (10  rows)
  products.csv     ──► bronze_products      (10  rows)
  transactions.csv ──► bronze_transactions  (2,000 rows)
  loans.csv        ──► bronze_loans         (500 rows)
```

In [ ]:
from pyspark.sql import functions as F

csv_sources = [
    ('branches',     'Files/branches.csv'),
    ('products',     'Files/products.csv'),
    ('customers',    'Files/customers.csv'),
    ('accounts',     'Files/accounts.csv'),
    ('transactions', 'Files/transactions.csv'),
    ('loans',        'Files/loans.csv'),
]

for table_name, file_path in csv_sources:
    df = spark.read.option('header','true').option('inferSchema','true').csv(file_path) \
        .withColumn('_ingested_at', F.current_timestamp()) \
        .withColumn('_source_file', F.lit(file_path))
    df.write.format('delta').mode('overwrite').saveAsTable(f'bronze_{table_name}')
    print(f'  bronze_{table_name:<20} {df.count():>5} rows')

In [ ]:
%%sql
SELECT 'bronze_branches'     AS Table, COUNT(*) AS Rows FROM bronze_branches     UNION ALL
SELECT 'bronze_products',              COUNT(*)         FROM bronze_products     UNION ALL
SELECT 'bronze_customers',             COUNT(*)         FROM bronze_customers    UNION ALL
SELECT 'bronze_accounts',              COUNT(*)         FROM bronze_accounts     UNION ALL
SELECT 'bronze_transactions',          COUNT(*)         FROM bronze_transactions UNION ALL
SELECT 'bronze_loans',                 COUNT(*)         FROM bronze_loans
ORDER BY Rows DESC